# 03 · Review lanes — the second opinions

Lane A (sorter_reviewer on exhausted medium classifications) and Lane B (judge_verify -> arbiter on ambiguous extractions).

## Setup — the lab bench

In [1]:
import json
import sys
from pathlib import Path

# Work from the repo root no matter where the kernel was started.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "notebooks" / "pipeline_lab.py").exists(), (
    f"llm-mailroom repo root not found above {Path.cwd()}"
)
sys.path.insert(0, str(ROOT / "notebooks"))

import pipeline_lab as lab


## What you'll see

- Lane A: reviewer agrees vs reviewer overrides
- the judge gate: extraction confidence strictly below 0.85 enters verification
- Lane B: a failed verdict arbitrated to the human siding
- the arbiter's bounded retry firing end-to-end (approve → re-extract → re-judge)

**Honesty label:** real graph, mock LLMs. The bounded-retry composition used to
dead-end silently (see below); since the KANBAN-098 fix it runs as designed,
and the regression is pinned by `src/tests/test_kanban098_arbiter_retry_path.py`.


## Lane A — agreement is NOT enough

A medium document that survives its retry meets the reviewer. If the reviewer merely agrees at medium confidence (0.88), the lane stays fail-safe: BOTH opinions ride along to the human siding. Only a HIGH-confidence answer (override or agreement, >= 0.95) extracts:

In [2]:
MED = lab.CLASSIFY_CONTRACT_MEDIUM          # 0.92, contract

with lab.lab_sandbox() as env:
    # Reviewer agrees with the sorter, but only at 0.88.
    lab.script_client(env["client"], reviewer=lab.REVIEWER_AGREE)
    r = lab.run_document(env, lab.DOC_CONTRACT, classification=MED,
                         extraction=lab.EXTRACT_HIGH,
                         filename="laneA_agree.txt")
    f = r["final"]
    print("agrees@0.88 :", f["review_verdict"], "->", f["stage"])
    print("  escalation:", f.get("escalation_reason"))


2026-09-01 23:02:00 [debug    ] score_configs_validated        count=59 registry=llm-dojo-scoring


agrees@0.88 : reviewer_agrees_low -> review
  escalation: sorter review: sorter='contract' (0.92) → reviewer='contract' (0.91, reviewer_agrees_low): MSA caption, defined terms, governing-law clause - a contract.


(The override arm — reviewer asserts court_opinion @ 0.97 and the document archives under the NEW type — was demonstrated in notebook 02.)

## The judge gate

Extraction confidence is checked against the judge band (strictly below 0.85 enters verification):

In [3]:
with lab.lab_sandbox() as env:
    for xc in (0.95, 0.80):
        r = lab.run_document(
            env, lab.DOC_CONTRACT,
            classification=lab.CLASSIFY_CONTRACT_HIGH,
            extraction={**lab.EXTRACT_HIGH, "confidence": xc},
            filename=f"gate_{int(xc * 100)}.txt",
        )
        p = lab.path_of(r["steps"])
        print(f"x={xc}: judge-verify {'RAN' if 'judge-verify' in p else 'skipped'}")


x=0.95: judge-verify RAN


x=0.8: judge-verify skipped


## Lane B — failed verdict, arbitrated away

The judge scores completeness. Below the bar, the arbiter decides. Here it declares the source itself ambiguous and escalates:

In [4]:
x93 = {**lab.EXTRACT_HIGH, "confidence": 0.93}

with lab.lab_sandbox() as env:
    lab.script_client(env["client"], judge=lab.JUDGE_PARTIAL,
                      arbiter=lab.ARBITER_HUMAN)
    r = lab.run_document(env, lab.DOC_CONTRACT,
                         classification=lab.CLASSIFY_CONTRACT_HIGH,
                         extraction=x93, filename="arb_human.txt")
    f = r["final"]
    print("judge:     ", f["judge_verdict"], f["judge_score"])
    print("arbiter:   ", f["arbiter_decision"])
    print("handoff:   ", f.get("arbiter_handoff"))
    print("stage:     ", f["stage"])


judge:      partial 0.55
arbiter:    human_review
handoff:    Escalating to the review siding
stage:      review


## Lane B — the bounded retry, end to end

The design: arbiter orders ONE re-extraction with a fix-list, the graph re-runs
the specialist, the judge verifies again. **A sharp edge lived here:** the
approving node sets `arbiter_retry_count` to 1 *at approval time* (the retrying
specialist reads it to weave the fix-list into its prompt), while the router
demanded `< 1` — so every approved retry escalated to humans instead of firing,
with both halves unit-green separately. Fixed under KANBAN-098: the bound is
approval-inclusive (`<= 1`) — first approval dispatches, a second arbitration
demanding another retry finds the budget spent. Watch the composed path run:


In [5]:
with lab.lab_sandbox() as env:
    # First judge call fails, second (post-retry) would pass — scripted
    # as a sequence; the last entry sticks.
    lab.script_client(env["client"],
                      judge=[lab.JUDGE_PARTIAL, lab.JUDGE_COMPLETE],
                      arbiter=lab.ARBITER_RETRY)
    r = lab.run_document(env, lab.DOC_CONTRACT,
                         classification=lab.CLASSIFY_CONTRACT_HIGH,
                         extraction=x93, filename="arb_retry.txt")
    f = r["final"]
    print("judge verdict: ", f["judge_verdict"])
    print("arbiter said:  ", f["arbiter_decision"])
    print("retry counter: ", f.get("arbiter_retry_count"))
    print("path:          ", " -> ".join(lab.path_of(r["steps"])))
    print("escalation:    ", f.get("escalation_reason"))


judge verdict:  complete
arbiter said:   retry_extraction
retry counter:  1
path:           intake-document -> classify-document -> extract-fields -> judge-verify -> arbitrate-verdict -> extract-fields -> judge-verify -> compile-report -> write-catalog -> archive-document
escalation:     arbiter ordered re-extraction: Re-extract with attention to §11 (governing law) and §14 (termination)


**Observed:** the arbiter approves, the counter reads 1, `retry_extract`
fires (second `extract-fields` span *after* `arbitrate-verdict`), the judge
scores the retried extraction complete, and the document compiles and archives.
Before KANBAN-098 this exact script dead-ended at the human siding — the fix
and its composed-path pins live in `src/tests/test_kanban098_arbiter_retry_path.py`.


## The state fields each lane leaves behind

In [6]:
fields = ["review_verdict", "reviewer_doc_type", "judge_verdict",
          "judge_score", "arbiter_decision", "arbiter_handoff",
          "arbiter_retry_count"]
for name in fields:
    print(f"{name:20s}", "tracked on DocumentState")


review_verdict       tracked on DocumentState
reviewer_doc_type    tracked on DocumentState
judge_verdict        tracked on DocumentState
judge_score          tracked on DocumentState
arbiter_decision     tracked on DocumentState
arbiter_handoff      tracked on DocumentState
arbiter_retry_count  tracked on DocumentState


## Where to go next

- **04 · human_in_the_loop** — the siding both lanes just used
- **02 · routing_dynamics** — how documents enter these lanes